# 03 Walk-Forward Cross-Validation

**Goal:** Honest OOS evaluation using walk-forward CV.

Three competing models trained and evaluated on **7 rolling windows** with a **3-month embargo** between training end and test month (eliminates label-horizon leakage):
- **HistGradientBoostingClassifier** (`class_weight='balanced'`) — main model
- **XGBClassifier** (`scale_pos_weight=7`) — GBM cross-check
- **LogisticRegression** (`class_weight='balanced'`) — interpretable baseline


In [1]:
import subprocess, sys
for pkg in ['feature-engine', 'scikit-learn', 'pyarrow', 'duckdb', 'xgboost']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Packages ready.')


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/zakaalzi/Desktop/talktalk/.venv/bin/python -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/zakaalzi/Desktop/talktalk/.venv/bin/python -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/zakaalzi/Desktop/talktalk/.venv/bin/python -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/zakaalzi/Desktop/talktalk/.venv/bin/python -m pip install --upgrade pip


Packages ready.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/zakaalzi/Desktop/talktalk/.venv/bin/python -m pip install --upgrade pip


In [2]:
import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy as np
import pandas as pd

from sklearn import set_config
set_config(transform_output='pandas')

from src.config import TRAIN_WINDOW_MONTHS, EMBARGO_MONTHS
from src.pipelines import ALL_FEATURES, make_boost_pipeline, make_xgb_pipeline, make_lr_pipeline
from src.evaluation import run_walk_forward_cv
print('Imports OK.')

Imports OK.


## 1. Load Master Table

In [3]:
master = pd.read_parquet('artifacts/master.parquet', engine='pyarrow')
master['datevalue'] = pd.to_datetime(master['datevalue'])

with open('artifacts/data_end.txt') as f:
    DATA_END = pd.Timestamp(f.read().strip())

print(f'master: {master.shape}  |  DATA_END: {DATA_END.date()}')
print(f'Label rate (all rows): {master["label"].mean():.3%}')
print(f'Label rate (clean):    {master[master["datevalue"] <= DATA_END]["label"].mean():.3%}')

master: (3292482, 69)  |  DATA_END: 2024-05-01
Label rate (all rows): 11.016%


Label rate (clean):    11.387%


## 2. Pipeline Definitions

The pipeline factories are defined in `src/pipelines.py`. The boost and XGB pipelines share the same head (`PackageGrouper → RareLabelEncoder → MeanEncoder → Winsorizer → model`). The LR pipeline is different: it uses `WoEEncoder` in place of MeanEncoder, plus `MeanMedianImputer → YeoJohnson → StandardScaler` before the model (LR requires scaled, imputed numeric inputs; boosters handle missing values natively).

Calling a factory (e.g. `make_boost_pipeline()`) returns a fresh unfitted `Pipeline` object, so each CV fold gets its own independent fit.

**Feature count:** `ALL_FEATURES` passes **40 columns** into the pipeline. `PackageGrouper` (step 0) replaces `crm_package_name` with three derived columns (`crm_package_grouped`, `package_tech`, `package_bundle_type`), so downstream steps and SHAP see **42 features**. The discrepancy is intentional and not a bug.

In [ ]:
# Sanity check: pipelines instantiate correctly — note the encoder difference
print('Pipeline steps — Boosting (also XGBoost):')
for name, step in make_boost_pipeline().steps:
    print(f'  {name}: {type(step).__name__}')

print('\nPipeline steps — Logistic Regression:')
for name, step in make_lr_pipeline().steps:
    print(f'  {name}: {type(step).__name__}')

print('\nAll features (pipeline inputs):', len(ALL_FEATURES))
print(ALL_FEATURES[:10], '...')

Pipeline steps — Boosting (also XGBoost):
  pkg: PackageGrouper
  rare: RareLabelEncoder
  encode: MeanEncoder
  winsor: Winsorizer
  model: HistGradientBoostingClassifier

Pipeline steps — Logistic Regression:
  pkg: PackageGrouper
  rare: RareLabelEncoder
  woe: WoEEncoder
  impute: MeanMedianImputer
  yj: YeoJohnsonTransformer
  scale: StandardScaler
  model: LogisticRegression

All features (pipeline inputs): 40
['crm_package_name', 'contract_status_risk', 'technology_group', 'sales_channel_group', 'tenure_days', 'ooc_days', 'speed', 'line_speed', 'speed_gap', 'calls_3m_count'] ...


## 3. Walk-Forward CV

```
Window 1 : train [Aug-22 … Jul-23]  →  gap [Aug-23 … Oct-23]  →  test Nov-23
Window 2 : train [Sep-22 … Aug-23]  →  gap [Sep-23 … Nov-23]  →  test Dec-23
...
Window 7 : train [Feb-23 … Jan-24]  →  gap [Feb-24 … Apr-24]  →  test May-24
```

The 3-month purge gap ensures no cease event that labels the test row can also label the last training row (90-day horizon = ~3 calendar months).

Each model is **refit from scratch** on the 12-month training slice. No data from
the test month is visible during training (including PackageGrouper's top-N selection).

In [5]:
window_results, obs_months = run_walk_forward_cv(
    master, ALL_FEATURES, TRAIN_WINDOW_MONTHS, DATA_END,
    make_boost=make_boost_pipeline,
    make_xgb=make_xgb_pipeline,
    make_lr=make_lr_pipeline,
    embargo_months=EMBARGO_MONTHS,
)

Valid months: 2022-08-01 -> 2024-05-01  (22 months)
Embargo: 3 month(s)  |  Expected windows: 7


  Aug22-Jul23 -> test Nov23  boost AUC=0.892 PR=0.636  XGB AUC=0.892 PR=0.635  LR AUC=0.859 PR=0.575  churn=12.22%


  Sep22-Aug23 -> test Dec23  boost AUC=0.889 PR=0.624  XGB AUC=0.889 PR=0.622  LR AUC=0.865 PR=0.572  churn=12.03%


  Oct22-Sep23 -> test Jan24  boost AUC=0.890 PR=0.646  XGB AUC=0.890 PR=0.644  LR AUC=0.867 PR=0.599  churn=12.66%


  Nov22-Oct23 -> test Feb24  boost AUC=0.889 PR=0.643  XGB AUC=0.888 PR=0.641  LR AUC=0.856 PR=0.591  churn=12.50%


  Dec22-Nov23 -> test Mar24  boost AUC=0.891 PR=0.638  XGB AUC=0.891 PR=0.639  LR AUC=0.857 PR=0.579  churn=12.22%


  Jan23-Dec23 -> test Apr24  boost AUC=0.896 PR=0.652  XGB AUC=0.895 PR=0.651  LR AUC=0.865 PR=0.597  churn=12.69%


  Feb23-Jan24 -> test May24  boost AUC=0.896 PR=0.655  XGB AUC=0.895 PR=0.652  LR AUC=0.875 PR=0.614  churn=12.15%

Completed 7 windows.


## 4. Save Results

In [6]:
import os, pickle
os.makedirs('artifacts', exist_ok=True)
with open('artifacts/window_results.pkl', 'wb') as f:
    pickle.dump({'window_results': window_results, 'obs_months': obs_months}, f)
print(f'Saved {len(window_results)} windows to artifacts/window_results.pkl')

# Quick headline
results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('y_test','boost_prob','xgb_prob','lr_prob')}
    for r in window_results
])
print(f'\nMean AUC  →  Boost: {results_df["boost_auc"].mean():.4f}  '
      f'XGB: {results_df["xgb_auc"].mean():.4f}  LR: {results_df["lr_auc"].mean():.4f}')
print(f'Mean PR   →  Boost: {results_df["boost_pr"].mean():.4f}  '
      f'XGB: {results_df["xgb_pr"].mean():.4f}  LR: {results_df["lr_pr"].mean():.4f}')

Saved 7 windows to artifacts/window_results.pkl

Mean AUC  →  Boost: 0.8920  XGB: 0.8913  LR: 0.8634
Mean PR   →  Boost: 0.6419  XGB: 0.6406  LR: 0.5898
